In [0]:
use catalog tpcds_lakehouse ;

## Dimention Table

In [0]:
CREATE TABLE IF NOT EXISTS silver.date_dim AS
SELECT
    d_date_sk,
    TRIM(d_date_id)                   AS d_date_id,
    d_date,
    d_month_seq,
    d_week_seq,
    d_quarter_seq,
    d_year,
    d_dow,
    d_moy,
    d_dom,
    d_qoy,
    d_fy_year,
    d_fy_quarter_seq,
    d_fy_week_seq,
    TRIM(d_day_name)                  AS d_day_name,
    TRIM(d_quarter_name)              AS d_quarter_name,
    d_holiday,
    d_weekend,
    d_following_holiday,
    d_first_dom,
    d_last_dom,
    d_same_day_ly,
    d_same_day_lq,
    d_current_day,
    d_current_week,
    d_current_month,
    d_current_quarter,
    d_current_year
FROM bronze.date_dim
WHERE d_date_sk IS NOT NULL;

In [0]:
-- Step 1: Create table once
CREATE TABLE IF NOT EXISTS silver.time_dim (
    t_time_sk   BIGINT,
    t_time_id   STRING,
    t_time      INT,
    t_hour      INT,
    t_minute    INT,
    t_second    INT,
    t_am_pm     STRING,
    t_shift     STRING,
    t_sub_shift STRING,
    t_meal_time STRING
);

-- Step 2: Run on every load
MERGE INTO silver.time_dim AS target
USING (
    SELECT
        t_time_sk,
        TRIM(t_time_id)   AS t_time_id,
        t_time,
        t_hour,
        t_minute,
        t_second,
        TRIM(t_am_pm)     AS t_am_pm,
        TRIM(t_shift)     AS t_shift,
        TRIM(t_sub_shift) AS t_sub_shift,
        TRIM(t_meal_time) AS t_meal_time
    FROM bronze.time_dim
    WHERE t_time_sk IS NOT NULL
) AS source
ON target.t_time_sk = source.t_time_sk
WHEN MATCHED AND (
    target.t_time_id   <> source.t_time_id   OR
    target.t_time      IS DISTINCT FROM source.t_time      OR
    target.t_hour      IS DISTINCT FROM source.t_hour      OR
    target.t_minute    IS DISTINCT FROM source.t_minute    OR
    target.t_second    IS DISTINCT FROM source.t_second    OR
    target.t_am_pm     IS DISTINCT FROM source.t_am_pm     OR
    target.t_shift     IS DISTINCT FROM source.t_shift     OR
    target.t_sub_shift IS DISTINCT FROM source.t_sub_shift OR
    target.t_meal_time IS DISTINCT FROM source.t_meal_time
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- Step 1: Create table once
CREATE TABLE IF NOT EXISTS silver.customer (
    c_customer_sk           BIGINT,
    c_customer_id           STRING,
    c_current_cdemo_sk      BIGINT,
    c_current_hdemo_sk      BIGINT,
    c_current_addr_sk       BIGINT,
    c_first_shipto_date_sk  BIGINT,
    c_first_sales_date_sk   BIGINT,
    c_salutation            STRING,
    c_first_name            STRING,
    c_last_name             STRING,
    c_preferred_cust_flag   STRING,
    c_birth_day             INT,
    c_birth_month           INT,
    c_birth_year            INT,
    c_birth_country         STRING,
    c_login                 STRING,
    c_email_address         STRING,
    c_last_review_date_sk   BIGINT
);

-- Step 2: Run on every load
MERGE INTO silver.customer AS target
USING (
    SELECT
        c_customer_sk,
        TRIM(c_customer_id)                AS c_customer_id,
        c_current_cdemo_sk,
        c_current_hdemo_sk,
        c_current_addr_sk,
        c_first_shipto_date_sk,
        c_first_sales_date_sk,
        TRIM(c_salutation)                 AS c_salutation,
        TRIM(c_first_name)                 AS c_first_name,
        TRIM(c_last_name)                  AS c_last_name,
        c_preferred_cust_flag,
        c_birth_day,
        c_birth_month,
        c_birth_year,
        NULLIF(TRIM(c_birth_country), '')  AS c_birth_country,
        NULLIF(TRIM(c_login), '')          AS c_login,
        NULLIF(TRIM(c_email_address), '')  AS c_email_address,
        c_last_review_date_sk
    FROM bronze.customer
    WHERE c_customer_sk IS NOT NULL
) AS source
ON target.c_customer_sk = source.c_customer_sk
WHEN MATCHED AND (
    target.c_customer_id          <> source.c_customer_id          OR
    target.c_current_cdemo_sk     IS DISTINCT FROM source.c_current_cdemo_sk     OR
    target.c_current_hdemo_sk     IS DISTINCT FROM source.c_current_hdemo_sk     OR
    target.c_current_addr_sk      IS DISTINCT FROM source.c_current_addr_sk      OR
    target.c_first_shipto_date_sk IS DISTINCT FROM source.c_first_shipto_date_sk OR
    target.c_first_sales_date_sk  IS DISTINCT FROM source.c_first_sales_date_sk  OR
    target.c_salutation           IS DISTINCT FROM source.c_salutation           OR
    target.c_first_name           IS DISTINCT FROM source.c_first_name           OR
    target.c_last_name            IS DISTINCT FROM source.c_last_name            OR
    target.c_preferred_cust_flag  IS DISTINCT FROM source.c_preferred_cust_flag  OR
    target.c_birth_day            IS DISTINCT FROM source.c_birth_day            OR
    target.c_birth_month          IS DISTINCT FROM source.c_birth_month          OR
    target.c_birth_year           IS DISTINCT FROM source.c_birth_year           OR
    target.c_birth_country        IS DISTINCT FROM source.c_birth_country        OR
    target.c_login                IS DISTINCT FROM source.c_login                OR
    target.c_email_address        IS DISTINCT FROM source.c_email_address        OR
    target.c_last_review_date_sk  IS DISTINCT FROM source.c_last_review_date_sk
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- Step 1: Create table once
CREATE TABLE IF NOT EXISTS silver.customer_demographics (
    cd_demo_sk              BIGINT,
    cd_gender               STRING,
    cd_marital_status       STRING,
    cd_education_status     STRING,
    cd_purchase_estimate    INT,
    cd_credit_rating        STRING,
    cd_dep_count            INT,
    cd_dep_employed_count   INT,
    cd_dep_college_count    INT
);

-- Step 2: Run on every load
MERGE INTO silver.customer_demographics AS target
USING (
    SELECT *
    FROM bronze.customer_demographics
    WHERE cd_demo_sk IS NOT NULL
) AS source
ON target.cd_demo_sk = source.cd_demo_sk
WHEN MATCHED AND (
    target.cd_gender               IS DISTINCT FROM source.cd_gender               OR
    target.cd_marital_status       IS DISTINCT FROM source.cd_marital_status       OR
    target.cd_education_status     IS DISTINCT FROM source.cd_education_status     OR
    target.cd_purchase_estimate    IS DISTINCT FROM source.cd_purchase_estimate    OR
    target.cd_credit_rating        IS DISTINCT FROM source.cd_credit_rating        OR
    target.cd_dep_count            IS DISTINCT FROM source.cd_dep_count            OR
    target.cd_dep_employed_count   IS DISTINCT FROM source.cd_dep_employed_count   OR
    target.cd_dep_college_count    IS DISTINCT FROM source.cd_dep_college_count
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- Step 1: Create table once
CREATE TABLE IF NOT EXISTS silver.customer_address (
    ca_address_sk    BIGINT,
    ca_address_id    STRING,
    ca_street_number STRING,
    ca_street_name   STRING,
    ca_street_type   STRING,
    ca_suite_number  STRING,
    ca_city          STRING,
    ca_county        STRING,
    ca_state         STRING,
    ca_zip           STRING,
    ca_country       STRING,
    ca_gmt_offset    DECIMAL(5,2),
    ca_location_type STRING
);

-- Step 2: Run on every load
MERGE INTO silver.customer_address AS target
USING (
    SELECT
        ca_address_sk,
        TRIM(ca_address_id)                    AS ca_address_id,
        NULLIF(TRIM(ca_street_number), '')     AS ca_street_number,
        NULLIF(TRIM(ca_street_name), '')       AS ca_street_name,
        NULLIF(TRIM(ca_street_type), '')       AS ca_street_type,
        NULLIF(TRIM(ca_suite_number), '')      AS ca_suite_number,
        NULLIF(TRIM(ca_city), '')              AS ca_city,
        NULLIF(TRIM(ca_county), '')            AS ca_county,
        NULLIF(TRIM(ca_state), '')             AS ca_state,
        NULLIF(TRIM(ca_zip), '')               AS ca_zip,
        NULLIF(TRIM(ca_country), '')           AS ca_country,
        ca_gmt_offset,
        NULLIF(TRIM(ca_location_type), '')     AS ca_location_type
    FROM bronze.customer_address
    WHERE ca_address_sk IS NOT NULL
) AS source
ON target.ca_address_sk = source.ca_address_sk
WHEN MATCHED AND (
    target.ca_address_id    <> source.ca_address_id    OR
    target.ca_street_number IS DISTINCT FROM source.ca_street_number OR
    target.ca_street_name   IS DISTINCT FROM source.ca_street_name   OR
    target.ca_street_type   IS DISTINCT FROM source.ca_street_type   OR
    target.ca_suite_number  IS DISTINCT FROM source.ca_suite_number  OR
    target.ca_city          IS DISTINCT FROM source.ca_city          OR
    target.ca_county        IS DISTINCT FROM source.ca_county        OR
    target.ca_state         IS DISTINCT FROM source.ca_state         OR
    target.ca_zip           IS DISTINCT FROM source.ca_zip           OR
    target.ca_country       IS DISTINCT FROM source.ca_country       OR
    target.ca_gmt_offset    IS DISTINCT FROM source.ca_gmt_offset    OR
    target.ca_location_type IS DISTINCT FROM source.ca_location_type
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- Step 1: Create table once (run only first time)
CREATE TABLE IF NOT EXISTS silver.item (
    i_item_sk        BIGINT,
    i_item_id        STRING,
    i_rec_start_date DATE,
    i_rec_end_date   DATE,
    i_item_desc      STRING,
    i_current_price  DECIMAL(7,2),
    i_wholesale_cost DECIMAL(7,2),
    i_brand_id       INT,
    i_brand          STRING,
    i_class_id       INT,
    i_class          STRING,
    i_category_id    INT,
    i_category       STRING,
    i_manufact_id    INT,
    i_manufact       STRING,
    i_size           STRING,
    i_formulation    STRING,
    i_color          STRING,
    i_units          STRING,
    i_container      STRING,
    i_manager_id     INT,
    i_product_name   STRING
);

-- Step 2: Run this on every load
MERGE INTO silver.item AS target
USING (
    SELECT
        i_item_sk,
        TRIM(i_item_id)               AS i_item_id,
        i_rec_start_date,
        i_rec_end_date,
        NULLIF(TRIM(i_item_desc), '') AS i_item_desc,
        i_current_price,
        i_wholesale_cost,
        i_brand_id,
        TRIM(i_brand)                 AS i_brand,
        i_class_id,
        TRIM(i_class)                 AS i_class,
        i_category_id,
        TRIM(i_category)              AS i_category,
        i_manufact_id,
        TRIM(i_manufact)              AS i_manufact,
        TRIM(i_size)                  AS i_size,
        TRIM(i_formulation)           AS i_formulation,
        TRIM(i_color)                 AS i_color,
        TRIM(i_units)                 AS i_units,
        TRIM(i_container)             AS i_container,
        i_manager_id,
        TRIM(i_product_name)          AS i_product_name
    FROM bronze.item
    WHERE i_item_sk IS NOT NULL
) AS source
ON target.i_item_sk = source.i_item_sk
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.store (
    s_store_sk             BIGINT,
    s_store_id             STRING,
    s_rec_start_date       DATE,
    s_rec_end_date         DATE,
    s_closed_date_sk       BIGINT,
    s_store_name           STRING,
    s_number_employees     INT,
    s_floor_space          INT,
    s_hours                STRING,
    s_manager              STRING,
    s_market_id            INT,
    s_geography_class      STRING,
    s_market_desc          STRING,
    s_market_manager       STRING,
    s_division_id          INT,
    s_division_name        STRING,
    s_company_id           INT,
    s_company_name         STRING,
    s_street_number        STRING,
    s_street_name          STRING,
    s_street_type          STRING,
    s_suite_number         STRING,
    s_city                 STRING,
    s_county               STRING,
    s_state                STRING,
    s_zip                  STRING,
    s_country              STRING,
    s_gmt_offset           DECIMAL(5,2),
    s_tax_precentage       DECIMAL(5,2)
);

MERGE INTO silver.store AS target
USING (SELECT * FROM bronze.store WHERE s_store_sk IS NOT NULL) AS source
ON target.s_store_sk = source.s_store_sk
WHEN MATCHED AND (
    target.s_store_id         IS DISTINCT FROM source.s_store_id         OR
    target.s_rec_start_date   IS DISTINCT FROM source.s_rec_start_date   OR
    target.s_rec_end_date     IS DISTINCT FROM source.s_rec_end_date     OR
    target.s_closed_date_sk   IS DISTINCT FROM source.s_closed_date_sk   OR
    target.s_store_name       IS DISTINCT FROM source.s_store_name       OR
    target.s_number_employees IS DISTINCT FROM source.s_number_employees OR
    target.s_floor_space      IS DISTINCT FROM source.s_floor_space      OR
    target.s_hours            IS DISTINCT FROM source.s_hours            OR
    target.s_manager          IS DISTINCT FROM source.s_manager          OR
    target.s_market_id        IS DISTINCT FROM source.s_market_id        OR
    target.s_geography_class  IS DISTINCT FROM source.s_geography_class  OR
    target.s_market_desc      IS DISTINCT FROM source.s_market_desc      OR
    target.s_market_manager   IS DISTINCT FROM source.s_market_manager   OR
    target.s_division_id      IS DISTINCT FROM source.s_division_id      OR
    target.s_division_name    IS DISTINCT FROM source.s_division_name    OR
    target.s_company_id       IS DISTINCT FROM source.s_company_id       OR
    target.s_company_name     IS DISTINCT FROM source.s_company_name     OR
    target.s_street_number    IS DISTINCT FROM source.s_street_number    OR
    target.s_street_name      IS DISTINCT FROM source.s_street_name      OR
    target.s_street_type      IS DISTINCT FROM source.s_street_type      OR
    target.s_suite_number     IS DISTINCT FROM source.s_suite_number     OR
    target.s_city             IS DISTINCT FROM source.s_city             OR
    target.s_county           IS DISTINCT FROM source.s_county           OR
    target.s_state            IS DISTINCT FROM source.s_state            OR
    target.s_zip              IS DISTINCT FROM source.s_zip              OR
    target.s_country          IS DISTINCT FROM source.s_country          OR
    target.s_gmt_offset       IS DISTINCT FROM source.s_gmt_offset       OR
    target.s_tax_precentage   IS DISTINCT FROM source.s_tax_precentage
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.promotion (
    p_promo_sk        BIGINT,
    p_promo_id        STRING,
    p_start_date_sk   BIGINT,
    p_end_date_sk     BIGINT,
    p_item_sk         BIGINT,
    p_cost            DECIMAL(15,2),
    p_response_target INT,              -- fixed
    p_promo_name      STRING,
    p_channel_dmail   STRING,
    p_channel_email   STRING,
    p_channel_catalog STRING,
    p_channel_tv      STRING,
    p_channel_radio   STRING,
    p_channel_press   STRING,
    p_channel_event   STRING,
    p_channel_demo    STRING,
    p_channel_details STRING,
    p_purpose         STRING,
    p_discount_active STRING
);

MERGE INTO silver.promotion AS target
USING (SELECT * FROM bronze.promotion WHERE p_promo_sk IS NOT NULL) AS source
ON target.p_promo_sk = source.p_promo_sk
WHEN MATCHED AND (
    target.p_promo_id        IS DISTINCT FROM source.p_promo_id        OR
    target.p_start_date_sk   IS DISTINCT FROM source.p_start_date_sk   OR
    target.p_end_date_sk     IS DISTINCT FROM source.p_end_date_sk     OR
    target.p_item_sk         IS DISTINCT FROM source.p_item_sk         OR
    target.p_cost            IS DISTINCT FROM source.p_cost            OR
    target.p_response_target IS DISTINCT FROM source.p_response_target OR  -- fixed
    target.p_promo_name      IS DISTINCT FROM source.p_promo_name      OR
    target.p_channel_dmail   IS DISTINCT FROM source.p_channel_dmail   OR
    target.p_channel_email   IS DISTINCT FROM source.p_channel_email   OR
    target.p_channel_catalog IS DISTINCT FROM source.p_channel_catalog OR
    target.p_channel_tv      IS DISTINCT FROM source.p_channel_tv      OR
    target.p_channel_radio   IS DISTINCT FROM source.p_channel_radio   OR
    target.p_channel_press   IS DISTINCT FROM source.p_channel_press   OR
    target.p_channel_event   IS DISTINCT FROM source.p_channel_event   OR
    target.p_channel_demo    IS DISTINCT FROM source.p_channel_demo    OR
    target.p_channel_details IS DISTINCT FROM source.p_channel_details OR
    target.p_purpose         IS DISTINCT FROM source.p_purpose         OR
    target.p_discount_active IS DISTINCT FROM source.p_discount_active
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.household_demographics (
    hd_demo_sk        BIGINT,
    hd_income_band_sk BIGINT,
    hd_buy_potential  STRING,
    hd_dep_count      INT,
    hd_vehicle_count  INT
);

MERGE INTO silver.household_demographics AS target
USING (SELECT * FROM bronze.household_demographics WHERE hd_demo_sk IS NOT NULL) AS source
ON target.hd_demo_sk = source.hd_demo_sk
WHEN MATCHED AND (
    target.hd_income_band_sk IS DISTINCT FROM source.hd_income_band_sk OR
    target.hd_buy_potential  IS DISTINCT FROM source.hd_buy_potential  OR
    target.hd_dep_count      IS DISTINCT FROM source.hd_dep_count      OR
    target.hd_vehicle_count  IS DISTINCT FROM source.hd_vehicle_count
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.web_site (
    web_site_sk        BIGINT,
    web_site_id        STRING,
    web_rec_start_date DATE,
    web_rec_end_date   DATE,
    web_name           STRING,
    web_open_date_sk   BIGINT,
    web_close_date_sk  BIGINT,
    web_class          STRING,
    web_manager        STRING,
    web_mkt_id         INT,
    web_mkt_class      STRING,
    web_mkt_desc       STRING,
    web_market_manager STRING,
    web_company_id     INT,
    web_company_name   STRING,
    web_street_number  STRING,
    web_street_name    STRING,
    web_street_type    STRING,
    web_suite_number   STRING,
    web_city           STRING,
    web_county         STRING,
    web_state          STRING,
    web_zip            STRING,
    web_country        STRING,
    web_gmt_offset     DECIMAL(5,2),
    web_tax_percentage DECIMAL(5,2)
);

MERGE INTO silver.web_site AS target
USING (SELECT * FROM bronze.web_site WHERE web_site_sk IS NOT NULL) AS source
ON target.web_site_sk = source.web_site_sk
WHEN MATCHED AND (
    target.web_site_id        IS DISTINCT FROM source.web_site_id        OR
    target.web_rec_start_date IS DISTINCT FROM source.web_rec_start_date OR
    target.web_rec_end_date   IS DISTINCT FROM source.web_rec_end_date   OR
    target.web_name           IS DISTINCT FROM source.web_name           OR
    target.web_open_date_sk   IS DISTINCT FROM source.web_open_date_sk   OR
    target.web_close_date_sk  IS DISTINCT FROM source.web_close_date_sk  OR
    target.web_class          IS DISTINCT FROM source.web_class          OR
    target.web_manager        IS DISTINCT FROM source.web_manager        OR
    target.web_mkt_id         IS DISTINCT FROM source.web_mkt_id         OR
    target.web_mkt_class      IS DISTINCT FROM source.web_mkt_class      OR
    target.web_mkt_desc       IS DISTINCT FROM source.web_mkt_desc       OR
    target.web_market_manager IS DISTINCT FROM source.web_market_manager OR
    target.web_company_id     IS DISTINCT FROM source.web_company_id     OR
    target.web_company_name   IS DISTINCT FROM source.web_company_name   OR
    target.web_street_number  IS DISTINCT FROM source.web_street_number  OR
    target.web_street_name    IS DISTINCT FROM source.web_street_name    OR
    target.web_street_type    IS DISTINCT FROM source.web_street_type    OR
    target.web_suite_number   IS DISTINCT FROM source.web_suite_number   OR
    target.web_city           IS DISTINCT FROM source.web_city           OR
    target.web_county         IS DISTINCT FROM source.web_county         OR
    target.web_state          IS DISTINCT FROM source.web_state          OR
    target.web_zip            IS DISTINCT FROM source.web_zip            OR
    target.web_country        IS DISTINCT FROM source.web_country        OR
    target.web_gmt_offset     IS DISTINCT FROM source.web_gmt_offset     OR
    target.web_tax_percentage IS DISTINCT FROM source.web_tax_percentage
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.web_page (
    wp_web_page_sk      BIGINT,
    wp_web_page_id      STRING,
    wp_rec_start_date   DATE,
    wp_rec_end_date     DATE,
    wp_creation_date_sk BIGINT,
    wp_access_date_sk   BIGINT,
    wp_autogen_flag     STRING,
    wp_customer_sk      BIGINT,
    wp_url              STRING,
    wp_type             STRING,
    wp_char_count       INT,
    wp_link_count       INT,
    wp_image_count      INT,
    wp_max_ad_count     INT
);

MERGE INTO silver.web_page AS target
USING (SELECT * FROM bronze.web_page WHERE wp_web_page_sk IS NOT NULL) AS source
ON target.wp_web_page_sk = source.wp_web_page_sk
WHEN MATCHED AND (
    target.wp_web_page_id      IS DISTINCT FROM source.wp_web_page_id      OR
    target.wp_rec_start_date   IS DISTINCT FROM source.wp_rec_start_date   OR
    target.wp_rec_end_date     IS DISTINCT FROM source.wp_rec_end_date     OR
    target.wp_creation_date_sk IS DISTINCT FROM source.wp_creation_date_sk OR
    target.wp_access_date_sk   IS DISTINCT FROM source.wp_access_date_sk   OR
    target.wp_autogen_flag     IS DISTINCT FROM source.wp_autogen_flag     OR
    target.wp_customer_sk      IS DISTINCT FROM source.wp_customer_sk      OR
    target.wp_url              IS DISTINCT FROM source.wp_url              OR
    target.wp_type             IS DISTINCT FROM source.wp_type             OR
    target.wp_char_count       IS DISTINCT FROM source.wp_char_count       OR
    target.wp_link_count       IS DISTINCT FROM source.wp_link_count       OR
    target.wp_image_count      IS DISTINCT FROM source.wp_image_count      OR
    target.wp_max_ad_count     IS DISTINCT FROM source.wp_max_ad_count
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.warehouse (
    w_warehouse_sk   BIGINT,
    w_warehouse_id   STRING,
    w_warehouse_name STRING,
    w_warehouse_sq_ft INT,
    w_street_number  STRING,
    w_street_name    STRING,
    w_street_type    STRING,
    w_suite_number   STRING,
    w_city           STRING,
    w_county         STRING,
    w_state          STRING,
    w_zip            STRING,
    w_country        STRING,
    w_gmt_offset     DECIMAL(5,2)
);

MERGE INTO silver.warehouse AS target
USING (SELECT * FROM bronze.warehouse WHERE w_warehouse_sk IS NOT NULL) AS source
ON target.w_warehouse_sk = source.w_warehouse_sk
WHEN MATCHED AND (
    target.w_warehouse_id    IS DISTINCT FROM source.w_warehouse_id    OR
    target.w_warehouse_name  IS DISTINCT FROM source.w_warehouse_name  OR
    target.w_warehouse_sq_ft IS DISTINCT FROM source.w_warehouse_sq_ft OR
    target.w_street_number   IS DISTINCT FROM source.w_street_number   OR
    target.w_street_name     IS DISTINCT FROM source.w_street_name     OR
    target.w_street_type     IS DISTINCT FROM source.w_street_type     OR
    target.w_suite_number    IS DISTINCT FROM source.w_suite_number    OR
    target.w_city            IS DISTINCT FROM source.w_city            OR
    target.w_county          IS DISTINCT FROM source.w_county          OR
    target.w_state           IS DISTINCT FROM source.w_state           OR
    target.w_zip             IS DISTINCT FROM source.w_zip             OR
    target.w_country         IS DISTINCT FROM source.w_country         OR
    target.w_gmt_offset      IS DISTINCT FROM source.w_gmt_offset
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.ship_mode (
    sm_ship_mode_sk BIGINT,
    sm_ship_mode_id STRING,
    sm_type         STRING,
    sm_code         STRING,
    sm_carrier      STRING,
    sm_contract     STRING
);

MERGE INTO silver.ship_mode AS target
USING (SELECT * FROM bronze.ship_mode WHERE sm_ship_mode_sk IS NOT NULL) AS source
ON target.sm_ship_mode_sk = source.sm_ship_mode_sk
WHEN MATCHED AND (
    target.sm_ship_mode_id IS DISTINCT FROM source.sm_ship_mode_id OR
    target.sm_type         IS DISTINCT FROM source.sm_type         OR
    target.sm_code         IS DISTINCT FROM source.sm_code         OR
    target.sm_carrier      IS DISTINCT FROM source.sm_carrier      OR
    target.sm_contract     IS DISTINCT FROM source.sm_contract
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.income_band (
    ib_income_band_sk BIGINT,
    ib_lower_bound    INT,
    ib_upper_bound    INT
);

MERGE INTO silver.income_band AS target
USING (SELECT * FROM bronze.income_band WHERE ib_income_band_sk IS NOT NULL) AS source
ON target.ib_income_band_sk = source.ib_income_band_sk
WHEN MATCHED AND (
    target.ib_lower_bound IS DISTINCT FROM source.ib_lower_bound OR
    target.ib_upper_bound IS DISTINCT FROM source.ib_upper_bound
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.call_center (
    cc_call_center_sk   BIGINT,
    cc_call_center_id   STRING,
    cc_rec_start_date   DATE,
    cc_rec_end_date     DATE,
    cc_closed_date_sk   BIGINT,
    cc_open_date_sk     BIGINT,
    cc_name             STRING,
    cc_class            STRING,
    cc_employees        INT,
    cc_sq_ft            INT,
    cc_hours            STRING,
    cc_manager          STRING,
    cc_mkt_id           INT,
    cc_mkt_class        STRING,
    cc_mkt_desc         STRING,
    cc_market_manager   STRING,
    cc_division         INT,
    cc_division_name    STRING,
    cc_company          INT,
    cc_company_name     STRING,
    cc_street_number    STRING,
    cc_street_name      STRING,
    cc_street_type      STRING,
    cc_suite_number     STRING,
    cc_city             STRING,
    cc_county           STRING,
    cc_state            STRING,
    cc_zip              STRING,
    cc_country          STRING,
    cc_gmt_offset       DECIMAL(5,2),
    cc_tax_percentage   DECIMAL(5,2)
);

MERGE INTO silver.call_center AS target
USING (SELECT * FROM bronze.call_center WHERE cc_call_center_sk IS NOT NULL) AS source
ON target.cc_call_center_sk = source.cc_call_center_sk
WHEN MATCHED AND (
    target.cc_call_center_id IS DISTINCT FROM source.cc_call_center_id OR
    target.cc_rec_start_date IS DISTINCT FROM source.cc_rec_start_date OR
    target.cc_rec_end_date   IS DISTINCT FROM source.cc_rec_end_date   OR
    target.cc_closed_date_sk IS DISTINCT FROM source.cc_closed_date_sk OR
    target.cc_open_date_sk   IS DISTINCT FROM source.cc_open_date_sk   OR
    target.cc_name           IS DISTINCT FROM source.cc_name           OR
    target.cc_class          IS DISTINCT FROM source.cc_class          OR
    target.cc_employees      IS DISTINCT FROM source.cc_employees      OR
    target.cc_sq_ft          IS DISTINCT FROM source.cc_sq_ft          OR
    target.cc_hours          IS DISTINCT FROM source.cc_hours          OR
    target.cc_manager        IS DISTINCT FROM source.cc_manager        OR
    target.cc_mkt_id         IS DISTINCT FROM source.cc_mkt_id         OR
    target.cc_mkt_class      IS DISTINCT FROM source.cc_mkt_class      OR
    target.cc_mkt_desc       IS DISTINCT FROM source.cc_mkt_desc       OR
    target.cc_market_manager IS DISTINCT FROM source.cc_market_manager OR
    target.cc_division       IS DISTINCT FROM source.cc_division       OR
    target.cc_division_name  IS DISTINCT FROM source.cc_division_name  OR
    target.cc_company        IS DISTINCT FROM source.cc_company        OR
    target.cc_company_name   IS DISTINCT FROM source.cc_company_name   OR
    target.cc_street_number  IS DISTINCT FROM source.cc_street_number  OR
    target.cc_street_name    IS DISTINCT FROM source.cc_street_name    OR
    target.cc_street_type    IS DISTINCT FROM source.cc_street_type    OR
    target.cc_suite_number   IS DISTINCT FROM source.cc_suite_number   OR
    target.cc_city           IS DISTINCT FROM source.cc_city           OR
    target.cc_county         IS DISTINCT FROM source.cc_county         OR
    target.cc_state          IS DISTINCT FROM source.cc_state          OR
    target.cc_zip            IS DISTINCT FROM source.cc_zip            OR
    target.cc_country        IS DISTINCT FROM source.cc_country        OR
    target.cc_gmt_offset     IS DISTINCT FROM source.cc_gmt_offset     OR
    target.cc_tax_percentage IS DISTINCT FROM source.cc_tax_percentage
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS silver.catalog_page (
    cp_catalog_page_sk     BIGINT,
    cp_catalog_page_id     STRING,
    cp_start_date_sk       BIGINT,
    cp_end_date_sk         BIGINT,
    cp_department          STRING,
    cp_catalog_number      INT,
    cp_catalog_page_number INT,
    cp_description         STRING,
    cp_type                STRING
);

MERGE INTO silver.catalog_page AS target
USING (SELECT * FROM bronze.catalog_page WHERE cp_catalog_page_sk IS NOT NULL) AS source
ON target.cp_catalog_page_sk = source.cp_catalog_page_sk
WHEN MATCHED AND (
    target.cp_catalog_page_id     IS DISTINCT FROM source.cp_catalog_page_id     OR
    target.cp_start_date_sk       IS DISTINCT FROM source.cp_start_date_sk       OR
    target.cp_end_date_sk         IS DISTINCT FROM source.cp_end_date_sk         OR
    target.cp_department          IS DISTINCT FROM source.cp_department          OR
    target.cp_catalog_number      IS DISTINCT FROM source.cp_catalog_number      OR
    target.cp_catalog_page_number IS DISTINCT FROM source.cp_catalog_page_number OR
    target.cp_description         IS DISTINCT FROM source.cp_description         OR
    target.cp_type                IS DISTINCT FROM source.cp_type
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%python 
dbutils.jobs.taskValues.set(key="dim_tbl_value", value=2)